# dashboard objective loss panel v1

目标：验证 dashboard view model 和 HTML 页面可以从统一 `dashboard_export.json` 展示 `objective.loss.v1`，不直接读取散落实验 sidecar。

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

REPO = Path('/Users/rziiiii/dev/workspace')
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.dashboard import build_dashboard_view_model, write_dashboard_html

CARD = Path('research/dashboard_objective_loss_panel_v1')
SOURCE_EXPORT = Path('research/model_matrix_objective_loss_generation_v1/dashboard_export.json')
VIEW_MODEL_PATH = CARD / 'view_model.json'
HTML_PATH = CARD / 'dashboard.html'

CARD.mkdir(parents=True, exist_ok=True)
{
    'card': str(CARD),
    'source_export': str(SOURCE_EXPORT),
    'source_export_exists': SOURCE_EXPORT.exists(),
}

{'card': 'research/dashboard_objective_loss_panel_v1',
 'source_export': 'research/model_matrix_objective_loss_generation_v1/dashboard_export.json',
 'source_export_exists': True}

## 构建 view model

这里只读取 dashboard export payload；`objective_losses.json` 的真实文件路径只作为 export artifact 元数据存在，页面层不读取它。

In [2]:
payload = json.loads(SOURCE_EXPORT.read_text(encoding='utf-8'))
assert payload['objective']['losses']['schema_version'] == 'objective.loss.v1'
assert payload['objective']['losses']['reward_surface'] == 'retired'

view_model = build_dashboard_view_model(payload)
summary = view_model['objective_loss_summary']
assert summary['schema_version'] == 'dashboard.objective_loss_summary.v1'
assert summary['objective_loss_schema_version'] == 'objective.loss.v1'
assert summary['reward_surface'] == 'retired'
assert summary['loss_component_count'] == 4
assert 'artifacts' not in view_model
VIEW_MODEL_PATH.write_text(json.dumps(view_model, ensure_ascii=False, indent=2), encoding='utf-8')
{
    'view_model_schema': view_model['schema_version'],
    'objective_loss_summary_schema': summary['schema_version'],
    'top_loss_component': summary['top_loss_component'],
    'loss_components': summary['loss_components'],
}

{'view_model_schema': 'dashboard.view_model.v1',
 'objective_loss_summary_schema': 'dashboard.objective_loss_summary.v1',
 'top_loss_component': {'name': 'ranking_loss', 'value': 1.0},
 'loss_components': [{'name': 'mse_loss', 'value': 0.027420219441811285},
  {'name': 'classification_loss', 'value': 0.4358974358974359},
  {'name': 'ic_loss', 'value': 0.9773647976417846},
  {'name': 'ranking_loss', 'value': 1.0}]}

## 渲染 HTML 页面

页面应出现 Objective Loss 面板，并且不出现 `objective_losses.json` 的文件路径。

In [3]:
html = write_dashboard_html(view_model, output_path=HTML_PATH, title='SOXL Objective Loss Dashboard')
assert HTML_PATH.exists()
assert 'Objective Loss' in html
assert 'objective.loss.v1' in html
assert 'retired' in html
assert 'mse_loss' in html
assert 'ranking_loss' in html
assert str(payload['artifacts']['objective_losses']) not in html

run_summary = {
    'schema_version': 'research.dashboard_objective_loss_panel.v1',
    'status': 'completed',
    'source_export': str(SOURCE_EXPORT),
    'view_model': str(VIEW_MODEL_PATH),
    'dashboard_html': str(HTML_PATH),
    'view_model_schema_version': view_model['schema_version'],
    'objective_loss_summary_schema_version': summary['schema_version'],
    'objective_loss_schema_version': summary['objective_loss_schema_version'],
    'metric_source_schema_version': summary['metric_source_schema_version'],
    'reward_surface': summary['reward_surface'],
    'optimization_direction': summary['optimization_direction'],
    'loss_component_count': summary['loss_component_count'],
    'top_loss_component': summary['top_loss_component'],
    'artifact_path_hidden_from_html': str(payload['artifacts']['objective_losses']) not in html,
    'conclusion': 'dashboard view model 和 HTML 已从统一 export 展示 objective.loss.v1，不直接读取散落 sidecar。',
}
(CARD / 'analysis_summary.json').write_text(json.dumps(run_summary, ensure_ascii=False, indent=2), encoding='utf-8')
run_summary

{'schema_version': 'research.dashboard_objective_loss_panel.v1',
 'status': 'completed',
 'source_export': 'research/model_matrix_objective_loss_generation_v1/dashboard_export.json',
 'view_model': 'research/dashboard_objective_loss_panel_v1/view_model.json',
 'dashboard_html': 'research/dashboard_objective_loss_panel_v1/dashboard.html',
 'view_model_schema_version': 'dashboard.view_model.v1',
 'objective_loss_summary_schema_version': 'dashboard.objective_loss_summary.v1',
 'objective_loss_schema_version': 'objective.loss.v1',
 'metric_source_schema_version': 'objective.evaluation.v1',
 'reward_surface': 'retired',
 'optimization_direction': 'minimize',
 'loss_component_count': 4,
 'top_loss_component': {'name': 'ranking_loss', 'value': 1.0},
 'artifact_path_hidden_from_html': True,
 'conclusion': 'dashboard view model 和 HTML 已从统一 export 展示 objective.loss.v1，不直接读取散落 sidecar。'}

## 中文结论

- 做了什么：从 `dashboard_export.json` 构建 `dashboard.view_model.v1`，新增 `dashboard.objective_loss_summary.v1`，并渲染包含 Objective Loss 面板的 HTML 页面。
- 核心指标：页面展示 `objective.loss.v1`、`reward_surface=retired`、`optimization_direction=minimize`、4 个 loss component 和 top loss component。
- regime 差异：本实验只验证 objective loss 页面通路；regime/window 差异仍由模型卡、analysis overview 和 model difference 面板展示。
- 是否晋升：这是 `src/dashboard` 框架能力晋升，不是策略晋升。
- 风险：当前 loss 是矩阵级 summary，不是逐模型 loss 表；若后续需要按模型比较，需要在 `src/objective` 和 model matrix 生成 per-model loss table。
- 下一步：可以把 objective loss 状态加入 schedule notification，或扩展为 per-model objective loss 对比表。